# Multimodal AI with LFM3 and OpenVINO

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and uses development builds of Optimum Intel and OpenVINO. It may be fully supported and validated in the future.

[LFM3](https://www.liquid.ai/blog/lfm3) is a family of open-weight models from [Liquid AI](https://www.liquid.ai/) built on one shared backbone and designed for on-device use and domain-specific fine-tuning. The language model has a hybrid architecture that combines gated short convolutions with sliding-window and global attention layers, which keeps inference fast and memory-efficient on edge devices.

| Model | Input | Output | Highlights |
|---|---|---|---|
| **LFM3-1.7B** | Text | Text | Reasoning on/off per request, tool calling, structured outputs, 256K context |
| **LFM3-VL** | Text, images, video | Text | New in-house vision encoder, video understanding, thinking mode |
| **LFM3-Omni** | Text, images, video, audio | Text, speech | First LFM omni model: speech recognition, text-to-speech, spoken chat |

In this tutorial we consider how to run LFM3 models with OpenVINO:

1. Download preconverted OpenVINO models or convert them with [Optimum Intel](https://github.com/huggingface/optimum-intel).
2. Run **LFM3-1.7B** with [OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) `LLMPipeline`: direct answers vs. reasoning, multi-turn chat, tool calling and structured JSON output.
3. Run **LFM3-VL** with Optimum Intel `OVModelForVisualCausalLM`: image understanding, multi-image comparison, thinking mode and video understanding.
4. Run **LFM3-Omni** with Optimum Intel `OVModelForMultimodalLM`: text-to-speech, speech recognition, spoken questions about images and spoken chat.
5. Measure LFM3-1.7B performance with OpenVINO GenAI.
6. Launch an interactive demo with LFM3-Omni.

Each model section is self-contained: you can run only the parts you are interested in after the [Prerequisites](#Prerequisites) and [Prepare models](#Prepare-models) steps.

>**Note**: LFM3 models ship their modeling code on Hugging Face Hub, so model conversion and loading require `trust_remote_code=True`. Please review the model code before running it.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Prepare models](#Prepare-models)
- [Text generation with LFM3-1.7B](#Text-generation-with-LFM3-1.7B)
    - [Select inference device for LFM3-1.7B](#Select-inference-device-for-LFM3-1.7B)
    - [Run text generation](#Run-text-generation)
    - [Reasoning mode](#Reasoning-mode)
    - [Multi-turn chat](#Multi-turn-chat)
    - [Tool calling](#Tool-calling)
    - [Structured output](#Structured-output)
- [Visual understanding with LFM3-VL](#Visual-understanding-with-LFM3-VL)
    - [Select inference device for LFM3-VL](#Select-inference-device-for-LFM3-VL)
    - [Image understanding](#Image-understanding)
    - [Multi-image input](#Multi-image-input)
    - [Thinking mode](#Thinking-mode)
    - [Video understanding](#Video-understanding)
- [Omni-modal assistant with LFM3-Omni](#Omni-modal-assistant-with-LFM3-Omni)
    - [Select inference device for LFM3-Omni](#Select-inference-device-for-LFM3-Omni)
    - [Text-to-speech](#Text-to-speech)
    - [Speech recognition](#Speech-recognition)
    - [Spoken question about an image](#Spoken-question-about-an-image)
    - [Spoken chat](#Spoken-chat)
- [Measure LFM3-1.7B performance](#Measure-LFM3-1.7B-performance)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/lfm3/lfm3.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

LFM3 support is currently provided by [Optimum Intel PR #2087](https://github.com/huggingface/optimum-intel/pull/2087). The installation below uses that public PR until the support is merged and released.

Video decoding requires [TorchCodec](https://github.com/pytorch/torchcodec#installing-torchcodec), installed below, and compatible FFmpeg shared libraries. Installing `av` alone does not enable video-file input in Transformers 5.20. If you add TorchCodec to an already running notebook environment, restart the kernel before reloading Transformers and the model processor.

In [ ]:
%pip install -q -U pip
# Optimum Intel is installed before transformers: the transformers version required by LFM3 overrides its pin
# (a pip dependency warning is expected).
%pip install -q "git+https://github.com/huggingface/optimum-intel.git@refs/pull/2087/head" "nncf" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q -U --pre "openvino" "openvino-tokenizers" "openvino-genai" --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly
%pip install -q "transformers==5.20.0" "torch>=2.10" "torchvision>=0.25" "torchcodec" "av" "pillow" "librosa" "soundfile" "matplotlib" "gradio>=6.0" "huggingface_hub" --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
from pathlib import Path

import requests

for helper in ["notebook_utils.py", "cmd_helper.py"]:
    if not Path(helper).exists():
        r = requests.get(url=f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper}")
        open(helper, "w", encoding="utf-8").write(r.text)

if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/lfm3/gradio_helper.py")
    open("gradio_helper.py", "w", encoding="utf-8").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("lfm3.ipynb")

## Prepare models
[back to top ⬆️](#Table-of-contents:)

LFM3 models are converted to OpenVINO Intermediate Representation (IR) with [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) `optimum-cli`. The command needs the model id, the task (`text-generation-with-past` for the text model, `image-text-to-text` for the vision and omni models), the weight format and `--trust-remote-code`:

```
optimum-cli export openvino --model LiquidAI/LFM3-1.7B --task text-generation-with-past --weight-format int4 --group-size 64 --ratio 1.0 --trust-remote-code LFM3-1.7B/INT4
```

Weight compression with [NNCF](https://github.com/openvinotoolkit/nncf) reduces the model footprint: `INT8` keeps accuracy close to `FP16`, and `INT4` gives the smallest and fastest model.

For your convenience, preconverted models are published in the [OpenVINO collection on Hugging Face Hub](https://huggingface.co/OpenVINO). When **Use preconverted models** is selected, a ready-to-use model is downloaded if it is available, otherwise the model is converted locally.

INT4 conversion explicitly uses asymmetric group-wise compression with group size 64 and ratio 1.0, matching the prepared IRs.

To reuse your own converted models, set **Local IR root** to a folder containing `LFM3-1.7B-int4-ov`, `LFM3-1.7B-int8-ov`, etc. This takes priority over cached models and downloads. With a local root set, missing model files raise an error instead of starting a download or conversion. Leave the field empty to use the normal download/conversion path.

In [ ]:
import ipywidgets as widgets

MODELS = {
    "text": {"model_id": "LiquidAI/LFM3-1.7B", "task": "text-generation-with-past"},
    "vision": {"model_id": "LiquidAI/LFM3-VL-2B", "task": "image-text-to-text"},
    "omni": {"model_id": "LiquidAI/LFM3-Omni-2B", "task": "image-text-to-text"},
}

weight_format = widgets.Dropdown(options=["INT4", "INT8", "FP16"], value="INT4", description="Weight format:")
use_preconverted = widgets.Checkbox(value=True, description="Use preconverted models")
local_ir_root = widgets.Text(
    value="",
    description="Local IR root:",
    placeholder="Optional: folder containing <model-name>-<precision>-ov",
    layout=widgets.Layout(width="80%"),
)

widgets.VBox([weight_format, use_preconverted, local_ir_root])

In [ ]:
import huggingface_hub as hf_hub
from cmd_helper import optimum_cli


def prepare_model(kind):
    model_id = MODELS[kind]["model_id"]
    model_name = model_id.split("/")[-1]
    precision = weight_format.value
    if local_ir_root.value.strip():
        model_dir = Path(local_ir_root.value.strip()).expanduser() / f"{model_name}-{precision.lower()}-ov"
        model_stem = "openvino_model" if kind == "text" else "openvino_language_model"
        required_files = ["config.json", f"{model_stem}.xml", f"{model_stem}.bin"]
        if kind == "text":
            required_files += [
                "generation_config.json",
                "tokenizer_config.json",
                "openvino_tokenizer.xml",
                "openvino_tokenizer.bin",
                "openvino_detokenizer.xml",
                "openvino_detokenizer.bin",
            ]
        missing = [name for name in required_files if not (model_dir / name).is_file()]
        if missing:
            raise FileNotFoundError(f"Incomplete local {precision} {model_name} IR at {model_dir}. Missing: {', '.join(missing)}")
        print(f"Using local {precision} model: {model_dir}")
        return model_dir

    model_dir = Path(model_name) / precision
    if any(model_dir.glob("openvino_*model*.xml")):
        print(f"✅ {model_name} {precision} model is found in {model_dir}")
        return model_dir

    ov_model_id = f"OpenVINO/{model_name}-{precision.lower()}-ov"
    if use_preconverted.value and hf_hub.repo_exists(ov_model_id):
        print(f"⌛ Downloading {ov_model_id}")
        hf_hub.snapshot_download(ov_model_id, local_dir=model_dir)
    else:
        print(f"⌛ Converting {model_id} to OpenVINO IR with {precision} weights. It may take some time.")
        export_args = {"task": MODELS[kind]["task"], "weight-format": precision.lower(), "trust-remote-code": ""}
        if precision == "INT4":
            export_args.update({"group-size": 64, "ratio": 1.0})
        optimum_cli(model_id, model_dir, additional_args=export_args)
    print(f"✅ {model_name} {precision} model is available in {model_dir}")
    return model_dir


def split_reasoning(text):
    # LFM3 writes its reasoning before `</think>`; the opening `<think>` tag is a part of the prompt
    if "</think>" in text:
        reasoning, answer = text.split("</think>", 1)
        return reasoning.replace("<think>", "").strip(), answer.strip()
    return "", text.strip()


def load_image(url, filename):
    from PIL import Image

    path = Path(filename)
    if not path.exists():
        path.write_bytes(requests.get(url, timeout=60).content)
    return Image.open(path).convert("RGB")


CAT_IMAGE_URL = "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/d5fbbd1a-d484-415c-88cb-9986625b7b11"
SUNSET_IMAGE_URL = "https://github.com/user-attachments/assets/da3edb79-ae36-4973-9eaf-6ef712425faa"

## Text generation with LFM3-1.7B
[back to top ⬆️](#Table-of-contents:)

LFM3-1.7B is a text model tuned for conversation, reasoning, tool use and structured outputs. We run it with OpenVINO GenAI `LLMPipeline`, which loads the OpenVINO model together with the converted tokenizer and the chat template.

Conversations are passed to the pipeline as `ChatHistory`. Besides messages, `ChatHistory` keeps tool definitions and extra context variables for the chat template. LFM3 reasoning is enabled by default and can be switched off per request with the `enable_thinking` template variable.

In [ ]:
text_model_dir = prepare_model("text")

### Select inference device for LFM3-1.7B
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from notebook_utils import device_widget

text_device = device_widget(default="CPU", exclude=["NPU"])

text_device

### Run text generation
[back to top ⬆️](#Table-of-contents:)

The `generate_text` helper builds a `ChatHistory`, sets the `enable_thinking` flag and optional tools, and streams the generated text.

It starts from the model's generation configuration, preserving the release defaults: sampling with `temperature=0.1`, `top_k=50` and `repetition_penalty=1.1`. Structured JSON generation uses greedy decoding.

In [ ]:
import openvino_genai as ov_genai

text_pipe = ov_genai.LLMPipeline(str(text_model_dir), text_device.value)


def streamer(subword):
    print(subword, end="", flush=True)


def generate_text(messages, enable_thinking=False, tools=None, max_new_tokens=512, structured_output_config=None, stream=True):
    history = ov_genai.ChatHistory(messages)
    history.set_extra_context({"enable_thinking": enable_thinking})
    if tools:
        history.set_tools(tools)
    config = text_pipe.get_generation_config()
    config.max_new_tokens = max_new_tokens
    if structured_output_config is not None:
        config.do_sample = False
        config.structured_output_config = structured_output_config
    return text_pipe.generate(history, config, streamer if stream else None).texts[0]


answer = generate_text([{"role": "user", "content": "Explain how a heat pump works in three sentences."}])

### Reasoning mode
[back to top ⬆️](#Table-of-contents:)

With `enable_thinking=True` the model first reasons step by step and then gives the final answer. Reasoning helps on tasks that need several steps, at the cost of more generated tokens.

In [ ]:
question = "A bat and a ball cost 1.10 dollars in total. The bat costs 1.00 dollar more than the ball. How much does the ball cost?"

output = generate_text([{"role": "user", "content": question}], enable_thinking=True, max_new_tokens=1024, stream=False)
reasoning, answer = split_reasoning(output)
print(f"Reasoning:\n{reasoning}\n\nAnswer:\n{answer}")

### Multi-turn chat
[back to top ⬆️](#Table-of-contents:)

To continue a conversation, append the model answer and the next user message to the message list.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful travel assistant. Keep answers short."},
    {"role": "user", "content": "Suggest three things to do in Lisbon."},
]
answer = generate_text(messages)

messages.append({"role": "assistant", "content": answer})
messages.append({"role": "user", "content": "Which of them is the best on a rainy day?"})
print("\n" + "-" * 80)
answer = generate_text(messages)

### Tool calling
[back to top ⬆️](#Table-of-contents:)

LFM3 is post-trained for agentic workflows. Tool definitions in JSON schema format are passed with `ChatHistory.set_tools`, and the chat template adds them to the system prompt. When the model decides to call a tool, it generates a call in the following format:

```
<tool_call>
<function=get_weather>
<parameter=city>
Paris
</parameter>
</function>
</tool_call>
```

The application parses the calls, executes the tools and returns the results to the model as `tool` messages. Then the model writes the final answer.

In [ ]:
import json
import re

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Returns the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name, for example Paris"}},
                "required": ["city"],
            },
        },
    }
]


def get_weather(city):
    # A stub instead of a real weather service
    data = {"paris": (18, "sunny"), "london": (12, "rainy"), "boston": (9, "cloudy")}
    temperature, condition = data.get(city.lower(), (20, "clear"))
    return {"city": city, "temperature_c": temperature, "condition": condition}


TOOL_IMPLEMENTATIONS = {"get_weather": get_weather}
TOOL_CALL_RE = re.compile(r"<tool_call>\s*<function=([^\n>]+)>(.*?)</function>\s*</tool_call>", re.DOTALL)
PARAMETER_RE = re.compile(r"<parameter=([^>\n]+)>\n?(.*?)\n?</parameter>", re.DOTALL)


def parse_tool_calls(text):
    calls = []
    for name, body in TOOL_CALL_RE.findall(text):
        arguments = {}
        for key, value in PARAMETER_RE.findall(body):
            try:
                arguments[key] = json.loads(value)
            except json.JSONDecodeError:
                arguments[key] = value
        calls.append({"name": name.strip(), "arguments": arguments})
    return calls

In [ ]:
messages = [{"role": "user", "content": "What is the weather like in Paris and London right now? Where is it warmer?"}]
output = generate_text(messages, tools=tools)
tool_calls = parse_tool_calls(output)
print(f"\n\nParsed tool calls: {tool_calls}")

if tool_calls:
    content = split_reasoning(output)[1].split("<tool_call>")[0].strip()
    messages.append({"role": "assistant", "content": content, "tool_calls": [{"type": "function", "function": call} for call in tool_calls]})
    for call in tool_calls:
        result = TOOL_IMPLEMENTATIONS[call["name"]](**call["arguments"])
        messages.append({"role": "tool", "content": json.dumps(result)})
    print("\nFinal answer:")
    answer = generate_text(messages, tools=tools)

### Structured output
[back to top ⬆️](#Table-of-contents:)

OpenVINO GenAI can constrain generation with a JSON schema, so the output is always valid JSON that matches the schema. This is useful for information extraction and for integrating the model into applications.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "event": {"type": "string"},
        "date": {"type": "string"},
        "city": {"type": "string"},
        "participants": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["event", "date", "city", "participants"],
}

text = "Alice and Bob are going to a science fair in Berlin on Friday, March 6."
output = generate_text(
    [{"role": "system", "content": "Extract the event information."}, {"role": "user", "content": text}],
    structured_output_config=ov_genai.StructuredOutputConfig(json_schema=json.dumps(schema)),
    stream=False,
)
print(json.dumps(json.loads(output), indent=2))

Release the text model before loading the next one.

In [ ]:
import gc

del text_pipe
gc.collect();

## Visual understanding with LFM3-VL
[back to top ⬆️](#Table-of-contents:)

LFM3-VL combines the LFM3 language model with a new in-house hybrid vision encoder. Images and videos are processed at their native resolution, and the model supports thinking mode for harder visual questions.

In this section we use the Optimum Intel `OVModelForVisualCausalLM` class, which provides a Transformers-compatible `generate` API on top of the OpenVINO models. The Hugging Face processor prepares the prompt and the visual inputs.

In [ ]:
vl_model_dir = prepare_model("vision")

### Select inference device for LFM3-VL
[back to top ⬆️](#Table-of-contents:)

In [ ]:
vl_device = device_widget(default="CPU", exclude=["NPU"])

vl_device

In [ ]:
from transformers import AutoProcessor, TextStreamer
from optimum.intel.openvino import OVModelForVisualCausalLM

vl_processor = AutoProcessor.from_pretrained(vl_model_dir, trust_remote_code=True)
vl_model = OVModelForVisualCausalLM.from_pretrained(vl_model_dir, device=vl_device.value, trust_remote_code=True)


def generate_vl(content, images=None, videos=None, enable_thinking=False, max_new_tokens=512, stream=True):
    prompt = vl_processor.apply_chat_template(
        [{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    )
    inputs = vl_processor(text=[prompt], images=images, videos=videos, return_tensors="pt")
    streamer = TextStreamer(vl_processor.tokenizer, skip_prompt=True, skip_special_tokens=True) if stream else None
    output_ids = vl_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, streamer=streamer)
    return vl_processor.decode(output_ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True)

### Image understanding
[back to top ⬆️](#Table-of-contents:)

Each image in the message content is marked with `{"type": "image"}`, and the images themselves are passed to the processor in the same order.

In [ ]:
image = load_image(CAT_IMAGE_URL, "cat.png")
display(image)

answer = generate_vl([{"type": "image"}, {"type": "text", "text": "What is unusual in this picture?"}], images=[image])

In [ ]:
question = "Describe the spatial layout of the scene: where is the cat relative to the box and to the other objects?"
answer = generate_vl([{"type": "image"}, {"type": "text", "text": question}], images=[image])

### Multi-image input
[back to top ⬆️](#Table-of-contents:)

Several images can be combined in one request, for example to compare them.

In [ ]:
import matplotlib.pyplot as plt

image2 = load_image(SUNSET_IMAGE_URL, "sunset.png")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, img in zip(axes, [image, image2]):
    ax.imshow(img)
    ax.axis("off")
plt.show()

content = [
    {"type": "image"},
    {"type": "image"},
    {"type": "text", "text": "Compare these two images. What is shown in each of them and which one has a warmer color palette?"},
]
answer = generate_vl(content, images=[image, image2])

### Thinking mode
[back to top ⬆️](#Table-of-contents:)

LFM3-VL is a thinking model. With `enable_thinking=True` it reasons about the image before answering.

In [ ]:
question = "Could the cat comfortably fit inside the box? Explain your answer using what you can see."
output = generate_vl([{"type": "image"}, {"type": "text", "text": question}], images=[image], enable_thinking=True, max_new_tokens=1024, stream=False)
reasoning, answer = split_reasoning(output)
print(f"Reasoning:\n{reasoning}\n\nAnswer:\n{answer}")

### Video understanding
[back to top ⬆️](#Table-of-contents:)

LFM3-VL is the first LFM vision model that supports video. The processor samples frames from the video file, pairs consecutive frames and adds their timestamps to the prompt, so the model can reason about events over time.

This file-input path uses TorchCodec from the prerequisite installation. New torchvision versions no longer provide a video decoder; `av` is not an automatic fallback for this processor.

In [ ]:
from IPython.display import Video
from notebook_utils import download_file

video_path = Path("coco.mp4")
if not video_path.exists():
    download_file(
        "https://storage.openvinotoolkit.org/repositories/openvino_notebooks/data/data/video/Coco%20Walking%20in%20Berkeley.mp4",
        filename=video_path.name,
    )
display(Video(video_path, embed=True, width=480))

answer = generate_vl([{"type": "video"}, {"type": "text", "text": "Describe what happens in this video."}], videos=[str(video_path)], max_new_tokens=256)

Release the vision model before loading the next one.

In [ ]:
del vl_model
gc.collect();

## Omni-modal assistant with LFM3-Omni
[back to top ⬆️](#Table-of-contents:)

LFM3-Omni is one end-to-end model that understands text, images, video and audio and answers in text, speech, or text and speech interleaved. It replaces a pipeline of separate speech recognition, vision-language, language and text-to-speech models.

The task and the answer modality are controlled with the system prompt:

| Task | System prompt |
|---|---|
| Speech recognition | `Perform ASR in English.` |
| Text-to-speech | `Perform TTS in American English using a female voice.` |
| Spoken chat | `Respond with interleaved text and audio.` |
| Text answer to any input | `Respond with text.` |

Speech is generated as audio tokens that the bundled Liquid Codec decodes into a 24 kHz waveform. Optimum Intel `OVModelForMultimodalLM` runs all parts of the model with OpenVINO: with `return_audio=True`, `generate` returns the text tokens and the waveform.

In [ ]:
omni_model_dir = prepare_model("omni")

### Select inference device for LFM3-Omni
[back to top ⬆️](#Table-of-contents:)

In [ ]:
omni_device = device_widget(default="CPU", exclude=["NPU"])

omni_device

In [ ]:
import librosa
import numpy as np
import torch
from IPython.display import Audio
from optimum.intel.openvino import OVModelForMultimodalLM

omni_processor = AutoProcessor.from_pretrained(omni_model_dir, trust_remote_code=True)
omni_model = OVModelForMultimodalLM.from_pretrained(omni_model_dir, device=omni_device.value, trust_remote_code=True)

AUDIO_SAMPLING_RATE = omni_processor.feature_extractor.sampling_rate  # audio input, 16 kHz
SPEECH_SAMPLING_RATE = 24000  # speech output of the Liquid Codec


def generate_omni(messages, images=None, audio=None, speak=False, generation_mode=None, max_new_tokens=512, stream=True):
    prompt = omni_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    audio_kwargs = {"audio": [np.asarray(audio, dtype=np.float32)], "sampling_rate": AUDIO_SAMPLING_RATE} if audio is not None else {}
    inputs = omni_processor(text=[prompt], images=images, return_tensors="pt", **audio_kwargs)
    streamer = TextStreamer(omni_processor.tokenizer, skip_prompt=True, skip_special_tokens=True) if stream else None
    # speech tokens are always sampled, a fixed seed makes the result reproducible
    torch.manual_seed(0)
    if speak:
        output_ids, waveform = omni_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, return_audio=True, generation_mode=generation_mode, streamer=streamer
        )
    else:
        output_ids = omni_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, generation_mode=generation_mode or "text", streamer=streamer)
        waveform = None
    text = omni_processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1] :], skip_special_tokens=True)[0].strip()
    speech = waveform.squeeze().cpu().numpy() if waveform is not None else None
    return text, speech

### Text-to-speech
[back to top ⬆️](#Table-of-contents:)

With `generation_mode="audio"` the model answers with speech only.

In [ ]:
tts_text = "OpenVINO makes it easy to run multimodal models like LFM3 on your own laptop."
_, speech = generate_omni(
    [{"role": "system", "content": "Perform TTS in American English using a female voice."}, {"role": "user", "content": tts_text}],
    speak=True,
    generation_mode="audio",
    stream=False,
)
Audio(speech, rate=SPEECH_SAMPLING_RATE)

### Speech recognition
[back to top ⬆️](#Table-of-contents:)

The audio input is a 16 kHz mono waveform. Let's transcribe a recorded speech sample and the speech generated in the previous step.

In [ ]:
sample_path = Path("how_are_you_doing_today.wav")
if not sample_path.exists():
    download_file("https://storage.openvinotoolkit.org/models_contrib/speech/2021.2/librispeech_s5/how_are_you_doing_today.wav", filename=sample_path.name)

asr_messages = [{"role": "system", "content": "Perform ASR in English."}, {"role": "user", "content": [{"type": "audio"}]}]

sample, _ = librosa.load(sample_path, sr=AUDIO_SAMPLING_RATE, mono=True)
display(Audio(sample, rate=AUDIO_SAMPLING_RATE))
transcript, _ = generate_omni(asr_messages, audio=sample)

print("\n" + "-" * 80)
generated_speech = librosa.resample(speech, orig_sr=SPEECH_SAMPLING_RATE, target_sr=AUDIO_SAMPLING_RATE)
transcript, _ = generate_omni(asr_messages, audio=generated_speech)

### Spoken question about an image
[back to top ⬆️](#Table-of-contents:)

Inputs of different modalities can be combined in one turn. Here we ask a question about the image by voice: the question is synthesized by the model itself, and the answer is requested in text.

In [ ]:
question = "What animal is in this picture and what is it doing?"
_, question_speech = generate_omni(
    [{"role": "system", "content": "Perform TTS in American English using a male voice."}, {"role": "user", "content": question}],
    speak=True,
    generation_mode="audio",
    stream=False,
)
display(Audio(question_speech, rate=SPEECH_SAMPLING_RATE))

image = load_image(CAT_IMAGE_URL, "cat.png")
question_audio = librosa.resample(question_speech, orig_sr=SPEECH_SAMPLING_RATE, target_sr=AUDIO_SAMPLING_RATE)
answer, _ = generate_omni(
    [{"role": "system", "content": "Respond with text."}, {"role": "user", "content": [{"type": "image"}, {"type": "audio"}]}],
    images=[image],
    audio=question_audio,
)

### Spoken chat
[back to top ⬆️](#Table-of-contents:)

With the `Respond with interleaved text and audio.` system prompt the model answers with text and speech together.

In [ ]:
reply, speech = generate_omni(
    [{"role": "system", "content": "Respond with interleaved text and audio."}, {"role": "user", "content": "Tell me a fun fact about octopuses."}],
    speak=True,
)
if speech is not None:
    display(Audio(speech, rate=SPEECH_SAMPLING_RATE))

## Measure LFM3-1.7B performance
[back to top ⬆️](#Table-of-contents:)

OpenVINO GenAI collects performance metrics for every `generate` call. Let's measure the time to first token, the time per output token and the generation throughput of LFM3-1.7B for prompts of different length on the selected device. The numbers depend on your hardware, the device and the weight format.

This benchmark uses greedy decoding and a fixed output length, independently of the checkpoint's sampling defaults.

In [ ]:
text_model_dir = prepare_model("text")
text_pipe = ov_genai.LLMPipeline(str(text_model_dir), text_device.value)
tokenizer = text_pipe.get_tokenizer()

config = ov_genai.GenerationConfig()
config.do_sample = False
config.max_new_tokens = 128
config.ignore_eos = True
config.apply_chat_template = False

paragraph = "OpenVINO is an open-source toolkit for optimizing and deploying deep learning models on Intel hardware. "
tokens_per_paragraph = tokenizer.encode(paragraph).input_ids.shape[1]

text_pipe.generate(paragraph, config)  # warm-up

print(f"{'Input tokens':>12} | {'TTFT, ms':>9} | {'TPOT, ms':>9} | {'Throughput, tokens/s':>20}")
for target_tokens in [256, 1024, 4096]:
    prompt = paragraph * max(1, target_tokens // tokens_per_paragraph)
    # a list input returns DecodedResults with perf_metrics (a single string returns plain text)
    metrics = text_pipe.generate([prompt], config).perf_metrics
    print(f"{metrics.get_num_input_tokens():>12} | {metrics.get_ttft().mean:>9.1f} | {metrics.get_tpot().mean:>9.2f} | {metrics.get_throughput().mean:>20.2f}")

del text_pipe
gc.collect();

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Now you can talk to LFM3-Omni: upload an image, ask a question by text or by voice and get the answer as text or as text and speech.

In [ ]:
from gradio_helper import make_demo, to_mono_float32


def omni_chat(text, image, audio, speak):
    content = []
    if image is not None:
        content.append({"type": "image"})
    waveform = None
    if audio is not None:
        sampling_rate, waveform = to_mono_float32(audio)
        waveform = librosa.resample(waveform, orig_sr=sampling_rate, target_sr=AUDIO_SAMPLING_RATE)
        content.append({"type": "audio"})
    if text:
        content.append({"type": "text", "text": text})
    system_prompt = "Respond with interleaved text and audio." if speak else "Respond with text."
    reply, speech = generate_omni(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": content}],
        images=[image] if image is not None else None,
        audio=waveform,
        speak=speak,
        stream=False,
    )
    return reply, ((SPEECH_SAMPLING_RATE, speech) if speech is not None else None)


load_image(CAT_IMAGE_URL, "cat.png")
demo = make_demo(omni_chat, example_image="cat.png")

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/